In [2]:
# 必要なら一度だけ実行
!pip install open_clip_torch

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 13.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 2.6/2.6 MB 14.9 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# CLIP feature extraction

In [8]:
from pathlib import Path
import re
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

import open_clip

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [4]:
# Set the working directory
import os
import numpy as np
import pandas as pd

#work_dir = os.path.dirname(os.path.dirname(os.getcwd())) 
work_dir = os.path.dirname(os.getcwd())

print(f"Current working directory: {work_dir}")

Current working directory: c:\Users\dysk-\Desktop\Current task\EEG compe


In [18]:
DATA_DIR = Path(f"{work_dir}/data")
IMAGE_ROOT = Path(DATA_DIR / "training_images")

print("DATA_DIR exists:", DATA_DIR.exists())
print("IMAGE_ROOT exists:", IMAGE_ROOT.exists())

print("train image_paths exists:", (DATA_DIR / "train" / "image_paths.txt").exists())
print("val image_paths exists:", (DATA_DIR / "val" / "image_paths.txt").exists())

DATA_DIR exists: True
IMAGE_ROOT exists: True
train image_paths exists: True
val image_paths exists: True


In [19]:
def load_image_paths(split):
    txt_path = DATA_DIR / split / "image_paths.txt"

    lines = txt_path.read_text().splitlines()
    paths = [line.strip() for line in lines if len(line.strip()) > 0]

    print(f"[{split}] num sample paths:", len(paths))
    print(f"[{split}] first 10:")
    for p in paths[:10]:
        print(" ", p)

    return paths

In [20]:
train_paths = load_image_paths("train")
val_paths = load_image_paths("val")

print("train first image exists:", (IMAGE_ROOT / train_paths[0]).exists())
print("val first image exists:", (IMAGE_ROOT / val_paths[0]).exists())

print("train first full path:", IMAGE_ROOT / train_paths[0])
print("val first full path:", IMAGE_ROOT / val_paths[0])

[train] num sample paths: 118800
[train] first 10:
  00001_aardvark/aardvark_01b.jpg
  00001_aardvark/aardvark_01b.jpg
  00001_aardvark/aardvark_02s.jpg
  00001_aardvark/aardvark_02s.jpg
  00001_aardvark/aardvark_03s.jpg
  00001_aardvark/aardvark_03s.jpg
  00001_aardvark/aardvark_04s.jpg
  00001_aardvark/aardvark_04s.jpg
  00001_aardvark/aardvark_05s.jpg
  00001_aardvark/aardvark_05s.jpg
[val] num sample paths: 59400
[val] first 10:
  00001_aardvark/aardvark_01b.jpg
  00001_aardvark/aardvark_02s.jpg
  00001_aardvark/aardvark_03s.jpg
  00001_aardvark/aardvark_04s.jpg
  00001_aardvark/aardvark_05s.jpg
  00001_aardvark/aardvark_06s.jpg
  00001_aardvark/aardvark_07s.jpg
  00001_aardvark/aardvark_08s.jpg
  00001_aardvark/aardvark_09s.jpg
  00001_aardvark/aardvark_10s.jpg
train first image exists: True
val first image exists: True
train first full path: c:\Users\dysk-\Desktop\Current task\EEG compe\data\training_images\00001_aardvark\aardvark_01b.jpg
val first full path: c:\Users\dysk-\Deskt

In [21]:
def build_unique_mapping(paths):
    unique_paths = sorted(set(paths))
    path_to_idx = {p: i for i, p in enumerate(unique_paths)}
    inverse_indices = np.array([path_to_idx[p] for p in paths], dtype=np.int64)

    print("num sample paths:", len(paths))
    print("num unique paths:", len(unique_paths))
    print("first 10 unique paths:")
    for p in unique_paths[:10]:
        print(" ", p)
    print("inverse_indices shape:", inverse_indices.shape)

    return unique_paths, inverse_indices

In [22]:
train_unique_paths, train_inverse_indices = build_unique_mapping(train_paths)
val_unique_paths, val_inverse_indices = build_unique_mapping(val_paths)

print("train sample:", len(train_paths))
print("train unique:", len(train_unique_paths))

print("val sample:", len(val_paths))
print("val unique:", len(val_unique_paths))

num sample paths: 118800
num unique paths: 5940
first 10 unique paths:
  00001_aardvark/aardvark_01b.jpg
  00001_aardvark/aardvark_02s.jpg
  00001_aardvark/aardvark_03s.jpg
  00001_aardvark/aardvark_04s.jpg
  00001_aardvark/aardvark_05s.jpg
  00001_aardvark/aardvark_06s.jpg
  00001_aardvark/aardvark_07s.jpg
  00001_aardvark/aardvark_08s.jpg
  00001_aardvark/aardvark_09s.jpg
  00001_aardvark/aardvark_10s.jpg
inverse_indices shape: (118800,)
num sample paths: 59400
num unique paths: 5940
first 10 unique paths:
  00001_aardvark/aardvark_01b.jpg
  00001_aardvark/aardvark_02s.jpg
  00001_aardvark/aardvark_03s.jpg
  00001_aardvark/aardvark_04s.jpg
  00001_aardvark/aardvark_05s.jpg
  00001_aardvark/aardvark_06s.jpg
  00001_aardvark/aardvark_07s.jpg
  00001_aardvark/aardvark_08s.jpg
  00001_aardvark/aardvark_09s.jpg
  00001_aardvark/aardvark_10s.jpg
inverse_indices shape: (59400,)
train sample: 118800
train unique: 5940
val sample: 59400
val unique: 5940


In [23]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai",
    device=device,
)

clip_model.eval()

print("CLIP loaded")

open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

c:\Users\dysk-\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dysk-\.cache\huggingface\hub\models--timm--vit_base_patch32_clip_224.openai. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\dysk-\AppData\Local\Programs\Python\Python312\Lib\site-packages\ope

CLIP loaded


In [24]:
@torch.no_grad()
def extract_clip_features_unique(
    unique_paths,
    image_root,
    model,
    preprocess,
    device,
    batch_size=128,
):
    feats = []

    for i in tqdm(range(0, len(unique_paths), batch_size), desc="extract CLIP unique"):
        batch_paths = unique_paths[i:i + batch_size]

        imgs = []
        for rel_path in batch_paths:
            img_path = image_root / rel_path

            if not img_path.exists():
                raise FileNotFoundError(f"not found: {img_path}")

            img = Image.open(img_path).convert("RGB")
            img = preprocess(img)
            imgs.append(img)

        imgs = torch.stack(imgs, dim=0).to(device)

        z = model.encode_image(imgs)
        z = F.normalize(z, dim=1)

        feats.append(z.cpu().numpy().astype(np.float32))

    feats = np.concatenate(feats, axis=0)

    feats = feats / (
        np.linalg.norm(feats, axis=1, keepdims=True) + 1e-6
    )

    return feats.astype(np.float32)

In [25]:
def make_and_save_clip_features(split, paths, unique_paths, inverse_indices):
    print("=" * 80)
    print("split:", split)

    unique_feats = extract_clip_features_unique(
        unique_paths=unique_paths,
        image_root=IMAGE_ROOT,
        model=clip_model,
        preprocess=clip_preprocess,
        device=device,
        batch_size=128,
    )

    sample_feats = unique_feats[inverse_indices]
    sample_feats = sample_feats.astype(np.float32)
    sample_feats = sample_feats / (
        np.linalg.norm(sample_feats, axis=1, keepdims=True) + 1e-6
    )

    save_dir = DATA_DIR / split

    np.save(save_dir / "clip_unique_features.npy", unique_feats)
    np.save(save_dir / "clip_features.npy", sample_feats)
    np.save(save_dir / "clip_inverse_indices.npy", inverse_indices)

    print(f"[{split}] unique_feats:", unique_feats.shape, unique_feats.dtype)
    print(f"[{split}] sample_feats:", sample_feats.shape, sample_feats.dtype)
    print(f"[{split}] inverse_indices:", inverse_indices.shape, inverse_indices.dtype)
    print(f"[{split}] sample norm mean:", np.linalg.norm(sample_feats, axis=1).mean())

    print("saved:")
    print(" ", save_dir / "clip_unique_features.npy")
    print(" ", save_dir / "clip_features.npy")
    print(" ", save_dir / "clip_inverse_indices.npy")

    return unique_feats, sample_feats

In [26]:
train_unique_feats, train_clip_features = make_and_save_clip_features(
    split="train",
    paths=train_paths,
    unique_paths=train_unique_paths,
    inverse_indices=train_inverse_indices,
)

split: train


extract CLIP unique:   0%|          | 0/47 [00:00<?, ?it/s]

[train] unique_feats: (5940, 512) float32
[train] sample_feats: (118800, 512) float32
[train] inverse_indices: (118800,) int64
[train] sample norm mean: 0.9999991
saved:
  c:\Users\dysk-\Desktop\Current task\EEG compe\data\train\clip_unique_features.npy
  c:\Users\dysk-\Desktop\Current task\EEG compe\data\train\clip_features.npy
  c:\Users\dysk-\Desktop\Current task\EEG compe\data\train\clip_inverse_indices.npy


In [27]:
val_unique_feats, val_clip_features = make_and_save_clip_features(
    split="val",
    paths=val_paths,
    unique_paths=val_unique_paths,
    inverse_indices=val_inverse_indices,
)

split: val


extract CLIP unique:   0%|          | 0/47 [00:00<?, ?it/s]

[val] unique_feats: (5940, 512) float32
[val] sample_feats: (59400, 512) float32
[val] inverse_indices: (59400,) int64
[val] sample norm mean: 0.9999991
saved:
  c:\Users\dysk-\Desktop\Current task\EEG compe\data\val\clip_unique_features.npy
  c:\Users\dysk-\Desktop\Current task\EEG compe\data\val\clip_features.npy
  c:\Users\dysk-\Desktop\Current task\EEG compe\data\val\clip_inverse_indices.npy


In [28]:
check_paths = [
    DATA_DIR / "train" / "clip_unique_features.npy",
    DATA_DIR / "train" / "clip_features.npy",
    DATA_DIR / "train" / "clip_inverse_indices.npy",
    DATA_DIR / "val" / "clip_unique_features.npy",
    DATA_DIR / "val" / "clip_features.npy",
    DATA_DIR / "val" / "clip_inverse_indices.npy",
]

for p in check_paths:
    arr = np.load(p)
    print(p)
    print(" shape:", arr.shape)
    print(" dtype:", arr.dtype)
    if arr.ndim == 2:
        print(" norm mean:", np.linalg.norm(arr, axis=1).mean())
        print(" first row first 5:", arr[0, :5])
    else:
        print(" first 10:", arr[:10])
    print()

c:\Users\dysk-\Desktop\Current task\EEG compe\data\train\clip_unique_features.npy
 shape: (5940, 512)
 dtype: float32
 norm mean: 0.999999
 first row first 5: [-0.03017402 -0.04197101  0.01656424  0.00254068 -0.00262643]

c:\Users\dysk-\Desktop\Current task\EEG compe\data\train\clip_features.npy
 shape: (118800, 512)
 dtype: float32
 norm mean: 0.9999991
 first row first 5: [-0.03017402 -0.04197101  0.01656424  0.00254068 -0.00262643]

c:\Users\dysk-\Desktop\Current task\EEG compe\data\train\clip_inverse_indices.npy
 shape: (118800,)
 dtype: int64
 first 10: [0 0 1 1 2 2 3 3 4 4]

c:\Users\dysk-\Desktop\Current task\EEG compe\data\val\clip_unique_features.npy
 shape: (5940, 512)
 dtype: float32
 norm mean: 0.999999
 first row first 5: [-0.03017402 -0.04197101  0.01656424  0.00254068 -0.00262643]

c:\Users\dysk-\Desktop\Current task\EEG compe\data\val\clip_features.npy
 shape: (59400, 512)
 dtype: float32
 norm mean: 0.9999991
 first row first 5: [-0.03017402 -0.04197101  0.01656424  0.

In [29]:
train_eeg = np.load(DATA_DIR / "train" / "eeg.npy")
val_eeg = np.load(DATA_DIR / "val" / "eeg.npy")

train_clip = np.load(DATA_DIR / "train" / "clip_features.npy")
val_clip = np.load(DATA_DIR / "val" / "clip_features.npy")

print("train EEG:", train_eeg.shape)
print("train CLIP:", train_clip.shape)

print("val EEG:", val_eeg.shape)
print("val CLIP:", val_clip.shape)

assert len(train_eeg) == len(train_clip)
assert len(val_eeg) == len(val_clip)

print("OK: EEG samples and CLIP features match.")

train EEG: (118800, 17, 100)
train CLIP: (118800, 512)
val EEG: (59400, 17, 100)
val CLIP: (59400, 512)
OK: EEG samples and CLIP features match.
